In [11]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/northwind.sqlite")
db_connection = sqlite3.connect(db_path)

In [12]:
tables_query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

tables_df = pd.read_sql(tables_query, db_connection)

print(tables_df)

                    name
0             Categories
1        sqlite_sequence
2   CustomerCustomerDemo
3   CustomerDemographics
4              Customers
5              Employees
6    EmployeeTerritories
7          Order Details
8                 Orders
9               Products
10               Regions
11              Shippers
12             Suppliers
13           Territories


In [16]:
# Load data
sql_query = """
SELECT
p.ProductName,
od.Quantity,
p.UnitPrice,
o.CustomerID,
c.Country AS CustomerCountry,
s.SupplierID,
s.Country AS SupplierCountry,
o.OrderDate
FROM Products AS p
JOIN "Order Details" AS od ON p.ProductID = od.ProductID
JOIN Orders AS o ON od.OrderID = o.OrderID
JOIN Customers AS c ON o.CustomerID = c.CustomerID
JOIN Suppliers AS s ON p.SupplierID = s.SupplierID
"""
product_sales_df = pd.read_sql(sql = sql_query,con = db_connection)

In [17]:
product_sales_df.head()

,ProductName,Quantity,UnitPrice,CustomerID,CustomerCountry,SupplierID,SupplierCountry,OrderDate
0,Queso Cabrales,12,21.00,VINET,France,5,Spain,2016-07-04
1,Singaporean Hokkien Fried Mee,10,14.00,VINET,France,20,Singapore,2016-07-04
2,Mozzarella di Giovanni,5,34.80,VINET,France,14,Italy,2016-07-04
3,Tofu,9,23.25,TOMSP,Germany,6,Japan,2016-07-05
4,Manjimup Dried Apples,40,53.00,TOMSP,Germany,24,Australia,2016-07-05


In [ ]:
product_sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2155 entries, 0 to 2154
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ProductName      2155 non-null   object 
 1   Quantity         2155 non-null   int64  
 2   UnitPrice        2155 non-null   float64
 3   CustomerID       2155 non-null   object 
 4   CustomerCountry  2155 non-null   object 
 5   SupplierID       2155 non-null   int64  
 6   SupplierCountry  2155 non-null   object 
 7   OrderDate        2155 non-null   object 
dtypes: float64(1), int64(2), object(5)
memory usage: 134.8+ KB


In [28]:
#convert OrderDate to datetime
product_sales_df["OrderDate"] = pd.to_datetime(product_sales_df["OrderDate"])

In [29]:
#sort dataset
# Why sort? : Rolling calculations depend on chronological order

product_sales_df = product_sales_df.sort_values(["ProductName","OrderDate"])

In [30]:
#Grouped Rolling Mean: 
#Task: Calculate the 7-day rolling average quantity for each product.
sales_df = product_sales_df[
    ["ProductName", "OrderDate", "Quantity"]
].copy()


In [33]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2155 entries, 49 to 1879
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ProductName  2155 non-null   object        
 1   OrderDate    2155 non-null   datetime64[ns]
 2   Quantity     2155 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 67.3+ KB


##### Groupby Function:

A groupby operation involves some combination of splitting the object, applying a function, and combining the results.
This can be used to group large amounts of data and compute operations on these groups.


In [ ]:
sales_df_groupby = product_sales_df[["ProductName",  "Quantity"]]
sales_df_groupby.groupby("ProductName").rolling(window=7).mean()


Quantity
ProductName                  
Alice Mutton  49          NaN
              82          NaN
              124         NaN
              144         NaN
              187         NaN
...                       ...
Zaanse koeken 1542  26.000000
              1646  26.857143
              1765  25.857143
              1835  27.142857
              1879  24.285714

[2155 rows x 1 columns]

In [40]:
sales_df=sales_df.set_index("OrderDate")

In [42]:
grouped = sales_df.groupby("ProductName")
rolling_avg = (
    grouped["Quantity"]
    .rolling(window=7, min_periods=1)
    .mean()
)

print(rolling_avg.head(20))

ProductName   OrderDate 
Alice Mutton  2016-07-25    30.000000
              2016-08-13    22.500000
              2016-08-30    20.000000
              2016-09-10    25.000000
              2016-10-02    21.600000
              2016-10-25    21.333333
              2016-10-28    28.285714
              2016-11-05    29.142857
              2017-01-15    27.285714
              2017-01-30    31.571429
              2017-01-30    33.000000
              2017-02-12    33.285714
              2017-05-01    34.000000
              2017-05-08    29.714286
              2017-05-28    25.714286
              2017-06-10    27.714286
              2017-06-19    23.857143
              2017-07-22    31.000000
              2017-09-30    33.857143
              2017-10-08    33.142857
Name: Quantity, dtype: float64


In [47]:
#For all the stats
rolling_stats = (
    grouped["Quantity"]
    .rolling(window='7D', min_periods=1)   
    .agg(["count", "mean", "std", "min", "max", "sum"])
)
print(rolling_stats.head(20))

                         count   mean        std    min    max    sum
ProductName  OrderDate                                               
Alice Mutton 2016-07-25    1.0   30.0        NaN   30.0   30.0   30.0
             2016-08-13    1.0   15.0        NaN   15.0   15.0   15.0
             2016-08-30    1.0   15.0        NaN   15.0   15.0   15.0
             2016-09-10    1.0   40.0        NaN   40.0   40.0   40.0
             2016-10-02    1.0    8.0        NaN    8.0    8.0    8.0
             2016-10-25    1.0   20.0        NaN   20.0   20.0   20.0
             2016-10-28    2.0   45.0  35.355339   20.0   70.0   90.0
             2016-11-05    1.0   36.0        NaN   36.0   36.0   36.0
             2017-01-15    1.0    2.0        NaN    2.0    2.0    2.0
             2017-01-30    1.0   45.0        NaN   45.0   45.0   45.0
             2017-01-30    2.0   47.5   3.535534   45.0   50.0   95.0
             2017-02-12    1.0   10.0        NaN   10.0   10.0   10.0
             2017-05